In [ ]:
# 1. Unsloth aur required AI libraries install karein
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [6]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detect GPU capability
load_in_4bit = True # 4bit Quantization (Free GPU memory save karne ke liye)

# Llama-3 8B Base Model Load karna
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Training ke liye LoRA Adapters configure karna
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("✅ Model aur LoRA adapters successfully load ho gaye hain!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.3: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.10.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Model aur LoRA adapters successfully load ho gaye hain!


In [7]:
!pip install unsloth_zoo




In [8]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detect GPU capability
load_in_4bit = True # 4bit Quantization (Free GPU memory save karne ke liye)

# Llama-3 8B Base Model Load karna
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Training ke liye LoRA Adapters configure karna
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("✅ Model aur LoRA adapters successfully load ho gaye hain!")

==((====))==  Unsloth 2026.10.3: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


✅ Model aur LoRA adapters successfully load ho gaye hain!


In [10]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# 1. Apna School Q&A Dataset (Aap isme apne school ki baatein aur jod sakte hain)
dataset_data = [
    {"instruction": "School ka timing kya hai?", "output": "School subah 8:00 AM se dopahar 2:00 PM tak chalta hai."},
    {"instruction": "Principal ka naam kya hai?", "output": "School ke principal Dr. R. K. Sharma hain."},
    {"instruction": "Fee submit karne ki last date kya hai?", "output": "Har mahine ki 10 taarik tak fee submit karni hoti hai."},
    {"instruction": "Sports period kab hota hai?", "output": "Sports period har shanivar (Saturday) ko aakhri do periods mein hota hai."},
    {
        "instruction": "School ki timing kya hai?",
        "output": "School ki timing subah 8:00 AM se dopahar 2:00 PM tak hai, Monday se Saturday."
    },
    {
        "instruction": "Admission ke liye kya process hai aur last date kya hai?",
        "output": "Admission ke liye aap school office visit kar sakte hain ya online registration form bhar sakte hain. Registration ki last date 31 March hai."
    },
    {
        "instruction": "Nursery ya KG mein admission ke liye child ki minimum age kitni honi chahiye?",
        "output": "Nursery mein admission ke liye bache ki umar 3 saal se adhik honi chahiye."
    },
    {
        "instruction": "Admission ke waqt kaun-kaun se documents lagte hain?",
        "output": "Admission ke liye birth certificate, aadhar card (child & parents), paasport size photos, aur pichhle school ka Report Card/TC (agar applicable ho) zaroori hai."
    },
    {
        "instruction": "Kya session ke beech mein (mid-term) admission mil sakta hai?",
        "output": "Haan, agar seats available hain aur student ke paas valid Transfer Certificate (TC) hai, toh mid-term admission mil sakta hai."
    },
    {
        "instruction": "Kya school mein sikhon (siblings) ke liye koi fee discount ya concession hai?",
        "output": "Haan, dusre ya chote bachhe ki tuition fee par 10% se 15% tak ka sibling concession diya jata hai."
    },
    {
        "instruction": "Registration fee kitni hoti hai?",
        "output": "Registration form ki fee ₹500 hai, jo ki non-refundable hoti hai."
    },
    {
        "instruction": "Kya merit-based scholarship milti hai?",
        "output": "Board exams ya annual exams mein top karne wale talented students ke liye merit scholarship ka pravadhan hai."
    },

    # --- Fee Structure & Payments ---
    {
        "instruction": "Class 1st se 5th tak ki monthly fee kitni hai?",
        "output": "Primary classes (1st to 5th) ki monthly tuition fee ₹3,500 hai."
    },
    {
        "instruction": "Class 6th se 10th tak ki fee structure batao?",
        "output": "Middle aur High school (6th to 10th) ki monthly tuition fee ₹5,000 hai."
    },
    {
        "instruction": "Class 11th aur 12th (Senior Secondary) ki fee kitni hai?",
        "output": "Class 11th aur 12th ki monthly fee stream par depend karti hai, jo ki roughly ₹6,000 se ₹7,000 per month hoti hai."
    },
    {
        "instruction": "Fee monthly bharni hoti hai ya quarterly?",
        "output": "Parents ke convenience ke liye fee quarterly (3 mahine ki ek sath) ya monthly dono options mein bhari ja sakti hai."
    },
    {
        "instruction": "Agar late fee bhari jaye toh kya fine lagta hai?",
        "output": "Due date ke baad fee bharne par ₹200 late fine charge kiya jata hai."
    },
    {
        "instruction": "Online fee payment kaise kar sakte hain?",
        "output": "Aap school ki official website ya mobile app par login karke Net Banking, UPI, ya Debit/Credit Card se fee pay kar sakte hain."
    },
    {
        "instruction": "Annual charges mein kya-kya include hota hai?",
        "output": "Annual charges mein library, generator backup, maintenance, aur annual sports/cultural events ka kharch shamil hota hai."
    },

    # --- Timings & Office Hours ---
    {
        "instruction": "Morning assembly ka time kya hota hai?",
        "output": "Morning assembly subah 8:00 AM par shuru hoti hai, isliye sabhi students ko 7:55 AM tak school pahunchna zaroori hai."
    },
    {
        "instruction": "Principal se milne ka samay kya hai?",
        "output": "Principal sir/mam se milne ka timing subah 10:00 AM se 12:00 PM tak hai (working days mein appointment ke sath)."
    },
    {
        "instruction": "School office kitne baje tak khula rehta hai?",
        "output": "School office subah 8:00 AM se shaam 4:00 PM tak khula rehta hai."
    },
    {
        "instruction": "Kya sardi ke mausam mein school ki timing badal jati hai?",
        "output": "Haan, winters (December-January) mein school ki timing subah 9:00 AM kar di jati hai."
    },
    {
        "instruction": "Student ki leave (chutti) ke liye application kaise deni hoti hai?",
        "output": "Leave ke liye parents diary mein note likh kar sign kar sakte hain ya school app ke zariye leave application bhej sakte hain."
    },

    # --- Academics & Examinations ---
    {
        "instruction": "School kis board se affiliated hai?",
        "output": "Hamara school CBSE board se affiliated hai aur modern NCERT curriculum follow karta hai."
    },
    {
        "instruction": "Class 11th mein kaun-kaun se streams available hain?",
        "output": "Class 11th ke liye Science (PCM/PCB), Commerce, aur Humanities (Arts) teeno streams available hain."
    },
    {
        "instruction": "Saal bhar mein kitne exams hote hain?",
        "output": "Academic session mein 4 unit tests, 1 Half-Yearly (Mid-Term) exam, aur 1 Annual exam hota hai."
    },
    {
        "instruction": "Parent-Teacher Meeting (PTM) kab hoti hai?",
        "output": "Har mahine ke 2nd Saturday ko ya har exam ke result ke baad PTM rakhi jati hai."
    },
    {
        "instruction": "Kya kamzor bachon ke liye extra classes milti hain?",
        "output": "Haan, weak students ke liye school hours ke baad ya Saturdays ko remedial (extra) classes lagayi jati hain."
    },
    {
        "instruction": "Home work kitna aur kaise milta hai?",
        "output": "Homework students ki school diary mein note karwaya jata hai aur school ke digital portal par bhi update hota hai."
    },

    # --- Facilities & Infrastructure ---
    {
        "instruction": "Kya school mein transport (bus) ki suvidha uplabdh hai?",
        "output": "Haan, school ki bus facility poore city ke major routes par available hai. Transport fee distance ke hisab se ₹1,500 se ₹2,500 per month hoti hai."
    },
    {
        "instruction": "Kya school buses mein GPS aur CCTV cameras lage hain?",
        "output": "Haan, sabhi school buses mein live GPS tracking aur security ke liye CCTV cameras installed hain."
    },
    {
        "instruction": "Library mein kitni books hain aur wahan baithne ki jagah hai?",
        "output": "Hamari library mein 5,000 se zyada books, encyclopedias, aur magazines hain jahan students shanti se padh sakte hain."
    },
    {
        "instruction": "Computer lab ki kya suvidha hai?",
        "output": "Hamare paas high-speed internet connection aur latest computers wali air-conditioned lab hai jahan har bache ko practicals milte hain."
    },
    {
        "instruction": "Science labs kaisi hain?",
        "output": "Physics, Chemistry, aur Biology ke liye alag-alag fully-equipped modern laboratories hain jahan safety ka pura dhyan rakha jata hai."
    },
    {
        "instruction": "Kya school mein canteen ya cafeteria hai?",
        "output": "Haan, school campus mein ek hygienic cafeteria hai jahan fresh aur healthy snacks milte hain."
    },
    {
        "instruction": "Medical emergency ya first-aid ki kya facility hai?",
        "output": "School mein ek dedicated Medical Room hai jisme first-aid kit aur ek qualified nurse hamesha available rehti hai."
    },
    {
        "instruction": "Kya school fully secure aur under CCTV surveillance hai?",
        "output": "Haan, poore campus aur classrooms mein 24/7 CCTV surveillance rehti hai aur trained security guards tainaat hain."
    },

    # --- Extracurricular Activities & Sports ---
    {
        "instruction": "School mein kaun-kaun se sports khele ja sakte hain?",
        "output": "School mein Cricket, Football, Basketball, Badminton, Volleyball, Table Tennis, aur Athletics ki professional training di jati hai."
    },
    {
        "instruction": "Indoor games ke liye kya arrangements hain?",
        "output": "Indoor games ke liye Chess, Carrom, Ludo, aur badhiya indoor activity hall available hai."
    },
    {
        "instruction": "Cultural activities mein kya sikhaya jata hai?",
        "output": "Students ko Classical/Western Dance, Vocal & Instrumental Music, Drawing, Painting, aur Drama/Theatre sikhaya jata hai."
    },
    {
        "instruction": "Annual Sports Day aur Annual Function kab hota hai?",
        "output": "Annual Sports Day December mein aur Annual Function February ke pehle haftay mein grand level par celebrate kiya jata hai."
    },
    {
        "instruction": "School mein kaun-kaun se clubs hain?",
        "output": "Eco Club, Science Club, Coding Club, Quiz Club, aur Literary Club hain jisme bache participate karte hain."
    },

    # --- Uniform & Rules ---
    {
        "instruction": "Summer aur winter uniform ka kya code hai?",
        "output": "Summer uniform mein boys ke liye light shirt aur dark trousers/shorts, aur girls ke liye tunic/skirt hoti hai. Winter uniform mein navy blue blazer aur pullover mandatory hai."
    },
    {
        "instruction": "ID Card aur school diary lana kitna zaroori hai?",
        "output": "Rozana school ID card aur school diary sath lana behad anivary hai, iske bina entry mein problem ho sakti hai."
    },
    {
        "instruction": "Mobile phones ya gadgets school laana allowed hai kya?",
        "output": "Nahi, students ke liye school campus mein mobile phones, smartwatches ya koi bhi electronic gadget lana sakht mana hai."
    },

    # --- Location & Address ---
    {
        "instruction": "School ka exact address kya hai?",
        "output": "School ka address Main City Road, Near Civil Lines hai."
    }
]

dataset = Dataset.from_list(dataset_data)

# 2. Llama-3 Prompt Template (Model ko samjhane ke liye ki sawal kya hai aur jawab kya dena hai)
prompt_style = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Model ko batane ke liye ki sentence kahan khatam hota hai

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = prompt_style.format(instruction, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

# 3. Trainer Setup karna
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30, # Chote dataset ke liye 30 steps kaafi hain
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Training shuru hone wali hai...")
trainer_stats = trainer.train()
print("🎉 Training successfully khatam ho gayi!")

Map:   0%|          | 0/47 [00:00<?, ? examples/s]

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/47 [00:00<?, ? examples/s]

🚀 Training shuru hone wali hai...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 47 | Num Epochs = 5 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,3.891016
2,3.759138
3,3.505748
4,3.429089
5,3.024830
6,2.345922
7,2.144532
8,1.605688
9,1.436073
10,1.500046


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-30/tokenizer_config.json.


🎉 Training successfully khatam ho gayi!


In [11]:
from unsloth import FastLanguageModel

# Inference (testing) ke liye model ko fast mode mein laana
FastLanguageModel.for_inference(model)

prompt_style = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

# Apne school se juda koi sawal yahan likhein (jo aapne training data mein diya tha)
question = "School ka timing kya hai?"

inputs = tokenizer(
    [
        prompt_style.format(question, "") # Instruction di, response ke liye jagah chhori
    ],
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
response = tokenizer.batch_decode(outputs, skip_special_tokens = True)

print(response[0])

Both `max_new_tokens` (=64) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
School ka timing kya hai?

### Response:
School subah 8:00 AM se dopahar 2:00 PM tak chalta hai.


In [12]:
from unsloth import FastLanguageModel

# Inference mode on karein
FastLanguageModel.for_inference(model)

# Apna koi bhi naya sawal yahan likhein
prompt = "Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nSchool ki timing kya hai?\n\n### Response:\n"

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=64) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
School ki timing kya hai?

### Response:
School subah 8:00 AM se dopahar 2:00 PM tak chalta hai.


In [13]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-i5afo80n/unsloth_e5d707ce13d74be58e54bb8e536e2632
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-i5afo80n/unsloth_e5d707ce13d74be58e54bb8e536e2632
  Resolved https://github.com/unslothai/unsloth.git to commit 0edcd859161ffbee9d70e9334883cac72c90030d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [14]:
import gradio as gr
from unsloth import FastLanguageModel

# Model ko inference mode mein set karna
FastLanguageModel.for_inference(model)

prompt_style = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

# Chatbot ka function jo user ka sawal lega aur model se jawab nikal kar dega
def school_chatbot(message, history):
    inputs = tokenizer(
        [prompt_style.format(message, "")],
        return_tensors = "pt"
    ).to("cuda")

    outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
    response = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]

    # Response mein se sirf model ka jawab extract karne ke liye
    if "### Response:" in response:
        response = response.split("### Response:")[1].strip()

    return response

# Gradio ki madad se Chat UI launch karna
demo = gr.ChatInterface(
    fn = school_chatbot,
    title = "🏫 School AI Assistant",
    description = "Apne school se judi koi bhi baat poochein!"
)

# Colab par live public link generate karne ke liye
demo.launch(share = True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d5abde07979a4b5c54.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
